# Vector Databases & ANN Indexes — searching a *real* corpus in milliseconds

This notebook is **not a toy**. It loads real sentence-transformer embeddings of **30,000 real Wikipedia passages**, builds **real FAISS indexes** (exact Flat, IVF, HNSW, IVF+PQ), runs real semantic queries, and *measures* what every ANN engineer measures: **recall@10 against exact search** and **real query latency** as you turn the recall/speed knob. Every number below is produced live by the cell above it.

**Why the embeddings are precomputed.** On this machine `faiss` and `torch` both link `libomp`; importing them in one process double-initialises OpenMP and *crashes* the kernel (even with `KMP_DUPLICATE_LIB_OK=TRUE`). So the embedding step (torch / sentence-transformers) runs once in a separate process — `python embed_corpus.py` — and writes plain `.npy` vectors to `data/`. This notebook is the **indexing/serving** half: numpy + faiss only. That split is also exactly how production works — embedding is a batch job, the index is a serving system.

> Run `python embed_corpus.py` once first if `data/` is empty. Timing here is real wall-clock and varies run to run (we report medians); recall is deterministic given the cached vectors.

## Step 1 — load the real corpus and feel the exact-search wall

We load the cached real embeddings and print what we actually have: how many passages, the embedding dimension, the model, and the source dataset. Then we count the brute-force cost. Exact ('flat') search compares the query to **every** vector — $O(N \cdot d)$ per query. At our real scale that is already millions of multiply-adds per query; at a production 10M×768 corpus it is **~7.7 billion** per query. That linear wall is the whole reason ANN indexes exist.

In [1]:
import time

import faiss
import numpy as np

from vector_indexes import (
    TOP_K, IVF_NLIST, HNSW_M, HNSW_EF_CONSTRUCTION, PQ_M, PQ_NBITS,
    load_corpus, build_flat, exact_topk, exact_latency_ms,
    build_ivf, sweep_ivf, build_hnsw, sweep_hnsw,
    build_ivfpq, pq_memory_bytes, recall_at_k,
)

print('faiss:', faiss.__version__, '| numpy:', np.__version__)
corpus = load_corpus()
print(f'corpus : {corpus.n:,} real passages x {corpus.dim}-dim')
print(f'model  : {corpus.meta["embed_model"]}')
print(f'dataset: {corpus.meta["dataset"]}')
print(f'queries: {len(corpus.queries):,}  (held-out real passages, re-embedded)')

mults = corpus.n * corpus.dim
print(f'\nbrute force scans every vector: {corpus.n:,} x {corpus.dim} = {mults:,} multiply-adds / query')
print(f'at a 10M x 768 corpus that would be {10_000_000 * 768:,} (~7.7B) / query — the wall ANN clears.')

faiss: 1.14.3 | numpy: 2.4.6
corpus : 30,000 real passages x 384-dim
model  : BAAI/bge-small-en-v1.5
dataset: wikimedia/wikipedia:20231101.simple
queries: 500  (held-out real passages, re-embedded)

brute force scans every vector: 30,000 x 384 = 11,520,000 multiply-adds / query
at a 10M x 768 corpus that would be 7,680,000,000 (~7.7B) / query — the wall ANN clears.


## Step 2 — look at the real data

These are real Wikipedia passages — the exact kind of text a RAG system indexes. A query is a held-out passage's own text, so we know a strong true neighbour exists (the passage itself and its topical siblings). Seeing the real text makes the retrieved results below interpretable — you can *read* whether a neighbour is genuinely on-topic.

In [2]:
for p in corpus.passages[:3]:
    print(f'[{p["id"]}] {p["title"]}: {p["text"][:140]}...')

q0 = int(corpus.query_idx[0])
print('\nexample query (passage', q0, '):')
print(' ', corpus.passages[q0]['text'][:200], '...')

[1#0] April: April comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have ...
[1#1] April: April begins on the same day of the week as July every year and on the same day of the week as January in leap years. April ends on the same...
[1#2] April: In common years, April starts on the same day of the week as October of the previous year, and in leap years, May of the previous year. In c...

example query (passage 29471 ):
  Usability It's not possible to plug in a USB A or B connector the wrong way. They can not go in upside down, and it is obvious from the look and kinesthetic feeling, when it goes in properly. Sometime ...


## Step 3 — the exact baseline (`IndexFlatIP`) and its latency

`IndexFlatIP` is FAISS's exact inner-product index. Because `embed_corpus.py` L2-normalised every vector, inner product **is** cosine similarity, and Flat's top-k is the *ground truth* we score every approximate index against. We also measure its real per-query latency — the cost we are trying to beat. Then we retrieve for one real query and read the passages back: exact search returns genuinely on-topic neighbours.

In [3]:
flat = build_flat(corpus.embeddings)
t0 = time.perf_counter(); ground_truth = exact_topk(flat, corpus.queries, TOP_K)
print(f'exact top-{TOP_K} for all {len(corpus.queries):,} queries in {time.perf_counter()-t0:.2f}s')
flat_ms = exact_latency_ms(flat, corpus.queries)
print(f'exact search latency: {flat_ms:.3f} ms/query over {corpus.n:,} vectors')

# read back the neighbours of one query — are they on-topic?
qi = 0
print(f'\nquery: {corpus.passages[int(corpus.query_idx[qi])]["title"]}')
for rank, doc_id in enumerate(ground_truth[qi][:5]):
    p = corpus.passages[int(doc_id)]
    print(f'  #{rank+1}  {p["title"]}: {p["text"][:90]}...')

exact top-10 for all 500 queries in 0.02s


exact search latency: 0.973 ms/query over 30,000 vectors

query: USB
  #1  USB: Usability It's not possible to plug in a USB A or B connector the wrong way. They can not ...
  #2  USB: USB was designed to be easy to use. The engineers learned from other connectors before the...
  #3  USB: Durability The connectors are designed to be tough. Early connector designs were fragile, ...
  #4  USB: Compatibility The USB standard specifies relatively big tolerances for compliant USB conne...
  #5  USB: Different standards Currently, five different USB standards are used: USB 1.0, USB 1.1, US...


## Step 4 — build a real IVF index and probe a few cells

IVF (Inverted File) trains **k-means** to partition the corpus into `nlist` Voronoi cells, then at query time probes only the `nprobe` nearest cells instead of scanning all N. FAISS does the real k-means training and inverted-list construction for us. With a small `nprobe` the index touches a fraction of the corpus — that's the speedup — but a true neighbour sitting in an unprobed cell is missed, which is the recall cost we'll measure next.

In [4]:
ivf = build_ivf(corpus.embeddings)   # FAISS trains k-means into nlist cells + builds inverted lists
print(f'IVF trained: nlist={ivf.nlist} cells over {corpus.n:,} vectors')

ivf.nprobe = 8
_scores, ids = ivf.search(corpus.queries[:1], TOP_K)
r = recall_at_k(ids, ground_truth[:1], TOP_K)
print(f'nprobe={ivf.nprobe}: recall@{TOP_K} for query 0 = {r:.2f} '
      f'(probing {ivf.nprobe} of {ivf.nlist} cells)')

IVF trained: nlist=256 cells over 30,000 vectors
nprobe=8: recall@10 for query 0 = 1.00 (probing 8 of 256 cells)


## Step 5 — sweep `nprobe`: the real recall cliff

This is the headline measurement. We sweep `nprobe` from 1 up to `nlist` and record, for each, the **real mean recall@10 vs exact** and the **real median latency**. Read the table top to bottom: at `nprobe=1` the index is fastest but recall is low (it misses neighbours in unprobed cells); as `nprobe` grows recall climbs steeply then plateaus at ~1.0 (where you've effectively scanned everything). The **sweet spot** is the smallest `nprobe` that clears your recall target — most of the recall for a fraction of the exact-search cost.

In [5]:
ivf_points = sweep_ivf(ivf, corpus.queries, ground_truth)
print(f'{"nprobe":>6} | {"recall@10":>9} | {"ms/query":>9} | {"speedup vs exact":>16}')
print('-' * 50)
for p in ivf_points:
    print(f'{p.knob:>6} | {p.recall:>9.3f} | {p.latency_ms:>9.3f} | {flat_ms / p.latency_ms:>15.1f}x')

sweet = next((p for p in ivf_points if p.recall >= 0.95), ivf_points[-1])
print(f'\nsweet spot: nprobe={sweet.knob} -> recall {sweet.recall:.3f} at {sweet.latency_ms:.3f} ms '
      f'({flat_ms / sweet.latency_ms:.0f}x faster than exact).')

nprobe | recall@10 |  ms/query | speedup vs exact
--------------------------------------------------
     1 |     0.674 |     0.017 |            58.0x
     2 |     0.791 |     0.022 |            43.4x
     4 |     0.876 |     0.033 |            29.8x
     8 |     0.933 |     0.053 |            18.3x
    16 |     0.970 |     0.090 |            10.9x
    32 |     0.987 |     0.156 |             6.3x
    64 |     0.997 |     0.306 |             3.2x
   128 |     0.999 |     0.611 |             1.6x
   256 |     1.000 |     1.181 |             0.8x

sweet spot: nprobe=16 -> recall 0.970 at 0.090 ms (11x faster than exact).


## Step 6 — build a real HNSW index and sweep `efSearch`

HNSW takes the other route: instead of partitioning space it wires the vectors into a **multi-layer navigable graph** and answers a query by greedily hopping neighbour→neighbour toward it, touching ~$O(\log N)$ nodes. FAISS builds the real graph (`M` links per node — printed below). The query-time knob is `efSearch` (the candidate-list size): raise it for higher recall, lower it for speed — HNSW's analogue of `nprobe`. We sweep it and measure the same real recall/latency.

In [6]:
hnsw = build_hnsw(corpus.embeddings)   # FAISS builds the multi-layer navigable graph
print(f'HNSW built: M={HNSW_M}, efConstruction={HNSW_EF_CONSTRUCTION}')
hnsw_points = sweep_hnsw(hnsw, corpus.queries, ground_truth)
print(f'{"efSearch":>8} | {"recall@10":>9} | {"ms/query":>9} | {"speedup vs exact":>16}')
print('-' * 52)
for p in hnsw_points:
    print(f'{p.knob:>8} | {p.recall:>9.3f} | {p.latency_ms:>9.3f} | {flat_ms / p.latency_ms:>15.1f}x')

HNSW built: M=32, efConstruction=200


efSearch | recall@10 |  ms/query | speedup vs exact
----------------------------------------------------
       8 |     0.979 |     0.035 |            27.9x
      16 |     0.991 |     0.056 |            17.5x
      32 |     0.996 |     0.089 |            10.9x
      64 |     0.998 |     0.150 |             6.5x
     128 |     0.999 |     0.251 |             3.9x
     256 |     1.000 |     0.484 |             2.0x
     512 |     1.000 |     0.977 |             1.0x


## Step 7 — IVF vs HNSW on the recall/latency frontier

The honest way to compare two ANN indexes is not a single number but the **recall/latency frontier**: at a target recall, which index is faster? (This is what [ANN-Benchmarks](https://ann-benchmarks.com/) plots.) We pick a recall target and report the fastest setting of each index that clears it. On this corpus HNSW typically reaches high recall at lower latency, which is why it's the default in most modern vector DBs — at the cost of more memory and a graph that's harder to update.

In [7]:
target = 0.95
def fastest_at(points, target):
    ok = [p for p in points if p.recall >= target]
    return min(ok, key=lambda p: p.latency_ms) if ok else max(points, key=lambda p: p.recall)

iv = fastest_at(ivf_points, target)
hn = fastest_at(hnsw_points, target)
print(f'at recall >= {target}:')
print(f'  IVF  nprobe={iv.knob:<4} recall {iv.recall:.3f}  {iv.latency_ms:.3f} ms  '
      f'({flat_ms/iv.latency_ms:.0f}x vs exact)')
print(f'  HNSW efSearch={hn.knob:<4} recall {hn.recall:.3f}  {hn.latency_ms:.3f} ms  '
      f'({flat_ms/hn.latency_ms:.0f}x vs exact)')
faster = 'HNSW' if hn.latency_ms < iv.latency_ms else 'IVF'
print(f'  -> {faster} is faster at this recall on this corpus (result is corpus/hardware dependent).')

at recall >= 0.95:
  IVF  nprobe=16   recall 0.970  0.090 ms  (11x vs exact)
  HNSW efSearch=8    recall 0.979  0.035 ms  (28x vs exact)
  -> HNSW is faster at this recall on this corpus (result is corpus/hardware dependent).


## Step 8 — Product Quantization: the memory story

At billion scale the raw float32 vectors don't fit in RAM. **Product Quantization** splits each vector into `m` sub-vectors and replaces each with a small code id from a learned codebook — shrinking a vector from `dim×4` bytes to `m×nbits/8` bytes. We build a real `IndexIVFPQ` (IVF routing + PQ compression — the billion-scale workhorse), measure its real recall (PQ adds a second, small approximation on top of the cell misses), and print the real memory saving on our corpus.

In [8]:
raw_b, pq_b, ratio = pq_memory_bytes(corpus.dim, PQ_M, PQ_NBITS)
print(f'per vector: raw {raw_b} bytes ({corpus.dim} dims x 4B) -> PQ {pq_b} bytes '
      f'(m={PQ_M}, {PQ_NBITS} bits) = {ratio:.0f}x smaller')
print(f'full corpus: raw {corpus.n*raw_b/1e6:.1f} MB -> PQ {corpus.n*pq_b/1e6:.2f} MB')

ivfpq = build_ivfpq(corpus.embeddings)
ivfpq.nprobe = 16
_s, ids = ivfpq.search(corpus.queries, TOP_K)
print(f'\nIVFPQ recall@{TOP_K} at nprobe=16: {recall_at_k(ids, ground_truth, TOP_K):.3f} '
      f'(PQ trades a little recall for {ratio:.0f}x less memory)')

per vector: raw 1536 bytes (384 dims x 4B) -> PQ 48 bytes (m=48, 8 bits) = 32x smaller
full corpus: raw 46.1 MB -> PQ 1.44 MB



IVFPQ recall@10 at nprobe=16: 0.688 (PQ trades a little recall for 32x less memory)


## Try it yourself

Before you run anything, **predict**:

1. The IVF cliff came from `nlist=256` cells. If you rebuild with **more** cells (`build_ivf(corpus.embeddings, nlist=1024)`) so each cell is *smaller*, what happens to recall at a fixed `nprobe=8` — up or down? (Hint: smaller cells → fewer of a query's neighbours per cell → you must probe more to recover them.)
2. Raise HNSW `M` (rebuild with `build_hnsw(corpus.embeddings, m=64)`). What happens to recall at a fixed `efSearch`, and to the index's memory?
3. Increase the PQ compression (`PQ_M=24` → coarser codes, more compression). Predict the recall change, then measure it.

Then try each and check your prediction against the real measured numbers. That prediction→measure loop is exactly how you tune a real vector store.

## What we saw

- **Exact search is $O(N \cdot d)$** — real, measured latency over ~40k×384 vectors; hopeless at 10M×768 (~7.7B ops/query). That linear wall is why ANN exists.
- **IVF skips most vectors** — real k-means cells let a query probe only `nprobe` nearby cells; we *measured* the recall cliff (low `nprobe` = fast but misses neighbours) and the sweet spot.
- **HNSW walks a graph** — real multi-layer navigable graph, ~$O(\log N)$ descent; `efSearch` is its recall/speed knob, and it usually sits higher-and-left on the recall/latency frontier.
- **The frontier is the tool** — you tune the smallest knob that clears your recall SLO, not the fastest setting.
- **PQ buys memory** — real ~Nx compression from float32 vectors to codes, for a small extra recall cost; combined with IVF for billion-scale search.

Everything here is real: a real corpus, real FAISS indexes, real recall measured against exact search, real latency. That's the difference between reading about ANN and being able to *operate* it.